# 🎬 Wan 2.1 × FastAPI — Serveur de Rendu Vidéo IA

> **Prérequis** : Runtime → Change runtime type → **GPU (A100 ou T4)**
>
> Ajoutez vos secrets dans l'icône 🔑 à gauche :
> - `NGROK_TOKEN` — votre token ngrok (https://dashboard.ngrok.com)
> - `GROQ_API_KEY` — votre clé API Groq (optionnel, le frontend gère Groq)


In [ ]:
# ── Cellule 1 : Vérification GPU ──────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or '❌ Aucun GPU détecté — changez le runtime !')

In [ ]:
# ── Cellule 2 : Installation des dépendances ──────────────────────────────────
!pip install -q diffusers transformers accelerate sentencepiece
!pip install -q fastapi uvicorn[standard] pyngrok pillow numpy
!pip install -q realesrgan basicsr  # Pour l'upscaling 4×
print('✅ Dépendances installées')

In [ ]:
# ── Cellule 3 : Configuration ngrok ───────────────────────────────────────────
from google.colab import userdata
import os

NGROK_TOKEN = userdata.get('NGROK_TOKEN')

from pyngrok import ngrok, conf
conf.get_default().auth_token = NGROK_TOKEN

# Ouvrir le tunnel sur le port 8000
tunnel = ngrok.connect(8000, 'http')
NGROK_URL = tunnel.public_url
os.environ['NGROK_URL'] = NGROK_URL

print(f'✅ Tunnel ngrok actif !')
print(f'📋 URL à copier dans votre .env.local :')
print(f'   NEXT_PUBLIC_COLAB_URL={NGROK_URL}')

In [ ]:
# ── Cellule 4 : Écriture du serveur FastAPI ───────────────────────────────────
server_code = open('wan21_server.py').read() if os.path.exists('wan21_server.py') else None

if server_code is None:
    print('Téléchargement du serveur depuis GitHub...')
    !wget -q https://raw.githubusercontent.com/aivideonew36-cyber/ai-video-generator/main/colab/wan21_server.py

print('✅ Fichier serveur prêt')

In [ ]:
# ── Cellule 5 : Démarrage du serveur FastAPI (en arrière-plan) ────────────────
import subprocess, time, threading

def run_server():
    proc = subprocess.Popen(
        ['python', '-m', 'uvicorn', 'wan21_server:app', '--host', '0.0.0.0', '--port', '8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end='')

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(5)

print(f'✅ Serveur FastAPI démarré !')
print(f'🔗 Health check : {NGROK_URL}/health')
print()
print('═' * 60)
print(f'  NEXT_PUBLIC_COLAB_URL={NGROK_URL}')
print('═' * 60)
print('Collez cette URL dans votre .env.local Vercel !')

In [ ]:
# ── Cellule 6 (optionnel) : Test rapide de l'endpoint /health ─────────────────
import requests
r = requests.get(f'{NGROK_URL}/health', timeout=10)
print('Status:', r.status_code)
print(r.json())